# 03. 모델링

가이드 7~13단계: 가설 수립, 4가지 학습 전략 비교, 튜닝, 오류 분석, 개선 실험, 최종 평가

## 0. 공통 설정

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))   # notebooks/ 에서 src/ 를 import 하기 위함

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import ROOT, DATA_DIR, FIG_DIR, RESULT_DIR, CLASS_NAMES, set_seed, setup_korean_font, save_fig

SEED = 42
set_seed(SEED)
setup_korean_font()
print("프로젝트 루트:", ROOT)

프로젝트 루트: /Users/adj/Team_Project


In [2]:
import torch
from src.utils import get_device

device = get_device()
print("사용 디바이스:", device)
print("PyTorch 버전:", torch.__version__)

사용 디바이스: mps
PyTorch 버전: 2.15.0


## 7. 가설 수립

In [4]:
import torch.nn as nn
from torchvision import models
from torchvision.models import ResNet18_Weights

NUM_CLASSES = 5

def build_model(strategy):
    """전략에 맞게 ResNet18을 만들고, 학습할 층만 requires_grad=True로 남긴다."""
    weights = None if strategy == "scratch" else ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)   # 1000개 → 5개 출력으로 교체

    if strategy == "feature_extraction":          # 백본 고정, fc만 학습
        trainable = ("fc.",)
    elif strategy == "partial_ft":                # 뒤쪽 layer4 + fc만 학습
        trainable = ("layer4.", "fc.")
    else:                                         # scratch, full_ft: 전체 학습
        trainable = None

    if trainable:
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith(trainable)
    return model

STRATEGIES = ["scratch", "feature_extraction", "partial_ft", "full_ft"]
rows = []
for s in STRATEGIES:
    m = build_model(s)
    total = sum(p.numel() for p in m.parameters())
    train_p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    rows.append({"전략": s, "초기 가중치": "무작위" if s == "scratch" else "ImageNet",
                 "전체 파라미터": total, "학습 파라미터": train_p,
                 "학습 비율(%)": round(train_p / total * 100, 2)})
param_df = pd.DataFrame(rows)
param_df

,전략,초기 가중치,전체 파라미터,학습 파라미터,학습 비율(%)
0,scratch,무작위,11179077,11179077,100.00
1,feature_extraction,ImageNet,11179077,2565,0.02
2,partial_ft,ImageNet,11179077,8396293,75.11
3,full_ft,ImageNet,11179077,11179077,100.00


### 현재 상황 (6단계)
- ImageNet과 대상(의류)은 유사하나, 분류 기준(소재 vs 형태)과 촬영 방식은 다름
- 데이터 양: Scratch에는 적고(이미지 1장당 파라미터 약 1,300개), 전이학습에는 중간

### 주 가설
현재 데이터는 ImageNet과 저수준 시각 특징(선·질감·원단)은 유사하지만 분류 기준이 되는 고수준 특징은 다르고,
학습 데이터는 백본 전체를 새로 학습하기엔 적으므로, 사전학습된 앞쪽 층은 고정하고 뒤쪽 layer4와 분류기만 학습하는
**Partial Fine-tuning**이 분류 성능과 학습 효율 측면에서 가장 유리할 것으로 예상한다.

### 전략별 예상
| 전략 | 학습 파라미터 | 예상 성능 | 예상 근거 |
|---|---|---|---|
| Scratch | 1,118만 (100%) | 가장 낮음, 과적합 가장 큼 | 보는 법부터 8,523장으로 학습 |
| Feature Extraction | 2,565 (0.02%) | 중간, 가장 빠름 | ImageNet 특징이 소재 기준으로 묶여 있어 스커트·데님 의류 구분에 한계 |
| Partial FT (layer4+fc) | 840만 (75%) | 가장 높음 | 저수준 특징 유지 + 고수준 분류 기준만 조정 |
| Full FT | 1,118만 (100%) | Partial과 비슷하거나 약간 낮음 | 조정 범위는 가장 넓지만 데이터 대비 과적합 위험 |

### 함께 확인할 세부 예측
- Feature Extraction은 스커트 Recall이 가장 낮고, 데님 재킷·스커트가 바지로 오분류될 것
- 모든 전략에서 조끼↔민소매 상의, 와이드 바지↔맥시 스커트에서 오분류가 몰릴 것 (3단계 비율 분석)
- Partial FT는 학습 파라미터가 75%지만, 계산이 무거운 앞쪽 층 역전파를 생략해 Epoch당 시간은 Full FT보다 짧을 것

## 8. 학습 전략 비교 (Scratch / Feature Extraction / Partial FT / Full FT)

In [8]:
%load_ext autoreload
%autoreload 2

import time
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from src.data import get_loaders

loaders = get_loaders(batch_size=32)
print({k: len(v.dataset) for k, v in loaders.items()})

{'train': 8523, 'val': 1722, 'test': 2597}


In [9]:
def set_train_mode(model):
    """학습 모드로 바꾸되, 고정한 층의 BatchNorm은 평가 모드로 유지한다."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not m.weight.requires_grad:
            m.eval()   # 고정된 층의 평균·분산 통계가 우리 데이터로 바뀌지 않게


def run_epoch(model, loader, criterion, optimizer=None):
    """optimizer가 있으면 학습 1 epoch, 없으면 평가만. 손실·정확도·Macro F1을 돌려준다."""
    is_train = optimizer is not None
    if is_train:
        set_train_mode(model)
    else:
        model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()        # 틀린 정도를 거꾸로 전달
                optimizer.step()       # 가중치 수정
            total_loss += loss.item() * x.size(0)
            all_preds.append(out.argmax(dim=1).cpu())
            all_labels.append(y.cpu())

    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return {"loss": total_loss / len(loader.dataset),
            "acc": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro")}


# 전략별 시작 학습률 (9단계 튜닝 전 출발점)
LR = {"scratch": 1e-3, "feature_extraction": 1e-3, "partial_ft": 1e-4, "full_ft": 1e-4}

def make_optimizer(model, strategy):
    params = [p for p in model.parameters() if p.requires_grad]   # 학습하는 숫자만 넘김
    return torch.optim.Adam(params, lr=LR[strategy])

In [10]:
criterion = nn.CrossEntropyLoss()
timing_rows = []

for s in STRATEGIES:
    set_seed(SEED)
    model = build_model(s).to(device)
    optimizer = make_optimizer(model, s)

    t0 = time.time()
    tr = run_epoch(model, loaders["train"], criterion, optimizer)
    t1 = time.time()
    va = run_epoch(model, loaders["val"], criterion)
    t2 = time.time()

    timing_rows.append({"전략": s, "학습(초)": round(t1 - t0), "검증(초)": round(t2 - t1),
                        "1 epoch 합계(초)": round(t2 - t0),
                        "train_acc": round(tr["acc"], 3), "val_acc": round(va["acc"], 3),
                        "val_macro_f1": round(va["macro_f1"], 3)})
    print(f"{s}: {t2 - t0:.0f}초 | val Macro F1 {va['macro_f1']:.3f}")

timing_df = pd.DataFrame(timing_rows)
timing_df.to_csv(RESULT_DIR / "epoch_time_check.csv", index=False)
timing_df

scratch: 157초 | val Macro F1 0.552
feature_extraction: 59초 | val Macro F1 0.841
partial_ft: 77초 | val Macro F1 0.921
full_ft: 149초 | val Macro F1 0.914


,전략,학습(초),검증(초),1 epoch 합계(초),train_acc,val_acc,val_macro_f1
0,scratch,146,10,157,0.636,0.557,0.552
1,feature_extraction,49,10,59,0.786,0.840,0.841
2,partial_ft,67,10,77,0.878,0.922,0.921
3,full_ft,139,10,149,0.875,0.914,0.914


In [11]:
import json

MAX_EPOCHS = 30
PATIENCE = 5
MODEL_DIR = RESULT_DIR / "models"          # .pt 파일은 .gitignore로 업로드 제외
MODEL_DIR.mkdir(exist_ok=True)

def train_strategy(strategy, max_epochs=MAX_EPOCHS, patience=PATIENCE, tag="base"):
    set_seed(SEED)
    model = build_model(strategy).to(device)
    optimizer = make_optimizer(model, strategy)
    history, best_f1, best_epoch, wait = [], -1.0, 0, 0
    t_start = time.time()

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        tr = run_epoch(model, loaders["train"], criterion, optimizer)
        va = run_epoch(model, loaders["val"], criterion)
        sec = time.time() - t0

        history.append({"epoch": epoch,
                        "train_loss": tr["loss"], "train_acc": tr["acc"], "train_macro_f1": tr["macro_f1"],
                        "val_loss": va["loss"], "val_acc": va["acc"], "val_macro_f1": va["macro_f1"],
                        "sec": sec})
        # 매 epoch 기록 저장: 밤사이 중간에 멈춰도 그때까지 결과가 남음
        pd.DataFrame(history).to_csv(RESULT_DIR / f"history_{tag}_{strategy}.csv", index=False)
        print(f"[{strategy}] {epoch:02d} | train loss {tr['loss']:.3f} acc {tr['acc']:.3f} "
              f"| val loss {va['loss']:.3f} F1 {va['macro_f1']:.3f} | {sec:.0f}s")

        if va["macro_f1"] > best_f1:                 # 최고 기록이면 모델 저장
            best_f1, best_epoch, wait = va["macro_f1"], epoch, 0
            torch.save(model.state_dict(), MODEL_DIR / f"{tag}_{strategy}.pt")
        else:
            wait += 1
            if wait >= patience:
                print(f"  → Early stopping (최고 epoch {best_epoch}, Val Macro F1 {best_f1:.3f})")
                break

    summary = {"tag": tag, "strategy": strategy, "best_epoch": best_epoch,
               "best_val_macro_f1": round(best_f1, 4), "epochs_run": len(history),
               "total_min": round((time.time() - t_start) / 60, 1),
               "lr": LR[strategy], "batch_size": 32, "max_epochs": max_epochs,
               "patience": patience, "device": str(device)}
    with open(RESULT_DIR / f"summary_{tag}_{strategy}.json", "w") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    del model
    if device.type == "mps":
        torch.mps.empty_cache()                     # 다음 전략을 위해 GPU 메모리 정리
    return summary

In [12]:
smoke = train_strategy("feature_extraction", max_epochs=2, tag="smoke")
print(smoke)
print(sorted(p.name for p in RESULT_DIR.iterdir()))

[feature_extraction] 01 | train loss 0.671 acc 0.776 | val loss 0.462 F1 0.842 | 66s
[feature_extraction] 02 | train loss 0.428 acc 0.846 | val loss 0.398 F1 0.862 | 74s
{'tag': 'smoke', 'strategy': 'feature_extraction', 'best_epoch': 2, 'best_val_macro_f1': 0.862, 'epochs_run': 2, 'total_min': 2.3, 'lr': 0.001, 'batch_size': 32, 'max_epochs': 2, 'patience': 5, 'device': 'mps'}
['.gitkeep', 'epoch_time_check.csv', 'history_smoke_feature_extraction.csv', 'models', 'summary_smoke_feature_extraction.json']


In [ ]:
all_summaries = []
for s in STRATEGIES:
    try:
        all_summaries.append(train_strategy(s))
    except Exception as e:
        print(f"!! {s} 실패: {e}")        # 한 전략이 실패해도 나머지는 계속 진행
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(RESULT_DIR / "summary_base_all.csv", index=False)
summary_df

## 9. 하이퍼파라미터 튜닝

(다음 단계에서 작성)

## 10~11. 오분류 분석 및 VLM 분석

(다음 단계에서 작성)

## 12. 개선 실험

(다음 단계에서 작성)

## 13. 최종 모델 Test 평가

(다음 단계에서 작성)